# EREC THRIVE — Climate-Health Risk Mapping
## Uganda Community Vulnerability Analysis

**Eco Reset Edge Connect (EREC)** | [ecoresetedge.org](https://ecoresetedge.org)

This notebook demonstrates the THRIVE Intelligence System's core vulnerability mapping capability — combining climate, health, and demographic data to generate community-level risk scores for Uganda.

### What this notebook covers
1. Data loading and exploration
2. Feature engineering — climate-health indicators
3. Vulnerability scoring model (Random Forest + SHAP)
4. Risk map visualisation (Folium/Leaflet)
5. District-level summary analysis
6. Early warning alert generation

---

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
import shap
from IPython.display import display

from models.vulnerability_scorer import VulnerabilityScorer, generate_sample_data, FEATURE_COLUMNS

# Plotting style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('✓ Libraries loaded')
print('✓ EREC THRIVE Intelligence System — ready')

## 1. Load Community Data

In production, this connects to DHIS2 and Kobo Toolbox APIs. Here we use synthetic data based on realistic Uganda district statistics.

In [ ]:
# Generate sample community dataset
df = generate_sample_data(n_communities=300, seed=42)

print(f'Communities: {len(df)}')
print(f'Districts: {df["district"].unique()}')
print(f'\nFeature summary:')
df[FEATURE_COLUMNS].describe().round(2)

## 2. Exploratory Analysis — Climate-Health Risk Patterns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Uganda Community Climate-Health Indicators', fontsize=14, fontweight='bold')

plot_cols = [
    ('flood_risk_score', 'Flood Risk Score (0-10)', '#2980b9'),
    ('malaria_incidence_rate', 'Malaria Incidence (per 1000)', '#c0392b'),
    ('air_quality_pm25', 'Air Quality PM2.5 (μg/m³)', '#8e44ad'),
    ('health_facility_distance_km', 'Distance to Health Facility (km)', '#16a085'),
    ('poverty_rate', 'Poverty Rate (%)', '#e67e22'),
    ('child_population_pct', 'Child Population (%)', '#27ae60'),
]

for ax, (col, label, color) in zip(axes.flat, plot_cols):
    ax.hist(df[col].dropna(), bins=25, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1, label=f'Mean: {df[col].mean():.1f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../docs/indicator_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to docs/indicator_distributions.png')

## 3. Vulnerability Scoring Model

In [ ]:
# Define vulnerability labels
# In production: derived from health outcome data and community assessments
labels = (
    (df['flood_risk_score'] > 6)
    | (df['malaria_incidence_rate'] > 100)
    | (df['poverty_rate'] > 50)
    | (df['health_facility_distance_km'] > 15)
).astype(int)

print(f'High vulnerability communities: {labels.sum()} / {len(labels)} ({labels.mean():.1%})')

# Train model
scorer = VulnerabilityScorer()
scorer.fit(df, labels)
print('\n✓ Model trained')

In [ ]:
# Score all communities
results = scorer.score(df)

print('Vulnerability score distribution:')
print(results['risk_category'].value_counts())
print(f'\nMean vulnerability score: {results["vulnerability_score"].mean():.1f}')
print(f'Max vulnerability score: {results["vulnerability_score"].max():.1f}')

# Score distribution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
colors_map = {'low': '#27ae60', 'moderate': '#f39c12', 'high': '#e67e22', 'critical': '#c0392b'}
ax1.hist(results['vulnerability_score'], bins=30, color='#2980b9', alpha=0.75, edgecolor='white')
ax1.axvline(30, color='#f39c12', linestyle='--', linewidth=1.5, label='Moderate threshold')
ax1.axvline(60, color='#e67e22', linestyle='--', linewidth=1.5, label='High threshold')
ax1.axvline(80, color='#c0392b', linestyle='--', linewidth=1.5, label='Critical threshold')
ax1.set_title('Vulnerability Score Distribution', fontweight='bold')
ax1.set_xlabel('Vulnerability Score (0-100)')
ax1.legend(fontsize=8)

# Category bar chart
cat_counts = results['risk_category'].value_counts().reindex(['critical', 'high', 'moderate', 'low'])
bars = ax2.bar(cat_counts.index, cat_counts.values, color=[colors_map[c] for c in cat_counts.index])
ax2.set_title('Communities by Risk Category', fontweight='bold')
ax2.set_ylabel('Number of communities')
for bar, val in zip(bars, cat_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/vulnerability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Feature Importance — What Drives Vulnerability?

In [ ]:
import shap

X_sample = scorer.preprocess(df.sample(100, random_state=42))
shap_values = scorer.explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=FEATURE_COLUMNS,
    plot_type='bar',
    show=False,
    max_display=10,
)
plt.title('Top 10 Drivers of Climate-Health Vulnerability', fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP analysis shows which factors most drive community vulnerability.')
print('This transparency is critical for community trust and local government action.')

## 5. Interactive Risk Map

In [ ]:
# Build interactive Folium map
m = folium.Map(location=[1.5, 32.5], zoom_start=7, tiles='OpenStreetMap')

risk_colors = {'critical': '#c0392b', 'high': '#e67e22', 'moderate': '#f39c12', 'low': '#27ae60'}

for _, row in results.iterrows():
    color = risk_colors.get(row['risk_category'], '#999')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5 + row['vulnerability_score'] / 15,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['community_name']}</b><br>"
            f"District: {row['district']}<br>"
            f"Score: {row['vulnerability_score']}<br>"
            f"Risk: <b>{row['risk_category'].upper()}</b><br>"
            f"Top factors: {', '.join(row['top_risk_factors'])}",
            max_width=200,
        ),
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;background:white;padding:10px;border-radius:6px;
            border:1px solid #ccc;font-size:12px;z-index:1000">
  <b>Vulnerability Risk</b><br>
  <span style="color:#c0392b">&#9679;</span> Critical (80-100)<br>
  <span style="color:#e67e22">&#9679;</span> High (60-80)<br>
  <span style="color:#f39c12">&#9679;</span> Moderate (30-60)<br>
  <span style="color:#27ae60">&#9679;</span> Low (0-30)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('../docs/uganda_vulnerability_map.html')
print('Interactive map saved to docs/uganda_vulnerability_map.html')
display(m)

## 6. Early Warning Alerts

In [ ]:
# Generate alerts for high/critical communities
alerts = results[results['risk_category'].isin(['high', 'critical'])].copy()
alerts = alerts.sort_values('vulnerability_score', ascending=False)

print(f'ACTIVE ALERTS: {len(alerts)} communities require immediate attention\n')
print('Top 10 Priority Communities:')
print('=' * 70)

for _, row in alerts.head(10).iterrows():
    risk_icon = '🔴' if row['risk_category'] == 'critical' else '🟠'
    print(f"{risk_icon} {row['community_name']:20s} | {row['district']:12s} | "
          f"Score: {row['vulnerability_score']:5.1f} | "
          f"Factors: {', '.join(row['top_risk_factors'][:2])}")

# District summary
print('\n\nDistrict-Level Alert Summary:')
print('=' * 50)
district_summary = alerts.groupby('district').agg(
    alert_count=('community_id', 'count'),
    critical=('risk_category', lambda x: (x=='critical').sum()),
    mean_score=('vulnerability_score', 'mean')
).round(1).sort_values('mean_score', ascending=False)
print(district_summary)

## 7. Export Results for DHIS2 / Kobo Integration

In [ ]:
# Export scored data in formats ready for humanitarian system integration
export_cols = [
    'community_id', 'community_name', 'district',
    'latitude', 'longitude',
    'vulnerability_score', 'risk_category',
    'flood_risk_score', 'malaria_incidence_rate',
    'poverty_rate', 'health_facility_distance_km',
]

# CSV export
results[export_cols].to_csv('../data/processed/community_vulnerability_scores.csv', index=False)

# GeoJSON export (for mapping tools and DHIS2)
import json
geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [row['longitude'], row['latitude']]},
            'properties': {col: row[col] for col in export_cols if col not in ['latitude', 'longitude']}
        }
        for _, row in results[export_cols].iterrows()
    ]
}

with open('../data/processed/community_vulnerability_scores.geojson', 'w') as f:
    json.dump(geojson, f, indent=2, default=str)

print('Exports complete:')
print('  ✓ data/processed/community_vulnerability_scores.csv')
print('  ✓ data/processed/community_vulnerability_scores.geojson')
print(f'  Total communities scored: {len(results)}')